In [3]:
import ollama
from deepeval.models.base_model import DeepEvalBaseLLM
class LocalLlama(DeepEvalBaseLLM):
   def __init__(self, model_name="llama3.2"):
       self.model_name = model_name
   def generate(self, prompt: str) -> str:
       response = ollama.generate(model=self.model_name, prompt=prompt)
       return response['response']

In [7]:
from ragas import evaluate
from ragas.metrics import _answer_relevance
local_model = "llama3.2"
test_case = {
   "input": "What is the capital of France?",
   "actual_output": "Nice is the capital of France.",
   "expected_output": "Paris is the capital of France."
}
metric = _answer_relevance(model=local_model,
                           input_key="input")
evaluate(test_cases=[test_case], metrics=[metric])

TypeError: 'module' object is not callable

In [ ]:
from datasets import Dataset 
import os
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness

data_samples = {
    'question': [
        'When was the first super bowl?', 
        'Who won the most super bowls?'
    ],
    'answer': [
        'The first superbowl was held on Jan 15, 1967', 
        'The most super bowls have been won by The New England Patriots'
    ],
    'contexts': [
        [
            'The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles,'
        ], 
        [
            'The Green Bay Packers...Green Bay, Wisconsin.',
            'The Packers compete...Football Conference'
        ]
    ],
    'ground_truth': [
        'The first superbowl was held on January 15, 1967', 
        'The New England Patriots have won the Super Bowl a record six times'
    ]
}

dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[faithfulness, answer_correctness])
df = score.to_pandas()
df.to_csv('score.csv', index=False)

In [ ]:
# Install dependencies before running:
# pip install llama-index ollama ragas datasets

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
#from llama_index.llms.ollama import Ollama
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas import evaluate
from datasets import Dataset
from langchain_ollama import ChatOllama


# -----------------------------
# 1. Load documents for RAG
# -----------------------------
# Put some .txt files in a "data" folder
documents = SimpleDirectoryReader("data").load_data()

# -----------------------------
# 2. Create a vector index
# -----------------------------
#llm = Ollama(model="llama3.2")  # You can use "mistral", "llama2", etc.
llm = ChatOllama(model="llama3.2:latest")
index = VectorStoreIndex.from_documents(documents)

# Create a query engine
query_engine = index.as_query_engine(llm=llm)

# -----------------------------
# 3. Define test queries
# -----------------------------
test_queries = [
    "What is the main topic of the document?",
    "Summarize the key points in two sentences."
]

# -----------------------------
# 4. Run RAG pipeline and collect results
# -----------------------------
questions = []
answers = []
contexts = []

for q in test_queries:
    response = query_engine.query(q)
    questions.append(q)
    answers.append(str(response))
    # Retrieve context chunks from the index
    retrieved_nodes = index.query(q).source_nodes
    contexts.append([node.get_content() for node in retrieved_nodes])

# -----------------------------
# 5. Prepare dataset for RAGAS
# -----------------------------
dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts
})

# -----------------------------
# 6. Evaluate with RAGAS
# -----------------------------
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

print("\n=== RAGAS Evaluation Results ===")
print(result)


In [ ]:
# Install dependencies before running:
# pip install ragas datasets ollama openai

from datasets import Dataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas import evaluate
from openai import OpenAI
import ollama
from langchain_ollama import ChatOllama

# --- Step 1: Configure Ollama as an OpenAI-compatible client ---
# Ollama must be running locally: `ollama serve`
# Example: `ollama run llama3`
client = OpenAI(
    base_url="http://localhost:11434/v1",  # Ollama's OpenAI-compatible API endpoint
    api_key="ollama"  # Dummy key; Ollama ignores it
)
llm = ChatOllama(model="llama3.2:latest")

# --- Step 2: Create a small synthetic RAG dataset ---
# Each entry contains: question, retrieved context, and generated answer
data = {
    "question": [
        "Who wrote the play Hamlet?",
        "What is the capital of France?"
    ],
    "contexts": [
        ["Hamlet is a tragedy written by William Shakespeare."],
        ["Paris is the capital city of France."]
    ],
    "answer": [
        "Hamlet was written by William Shakespeare.",
        "The capital of France is Paris."
    ],
    "reference": [
        "William Shakespeare is the author of Hamlet.",
        "Paris is the capital of France."
    ]
}

dataset = Dataset.from_dict(data)

# --- Step 3: Run RAGAS evaluation ---
# You can choose which metrics to run
metrics = [faithfulness, answer_relevancy, context_precision]

results = evaluate(dataset, metrics=metrics, llm=llm)

# --- Step 4: Display results ---
print("RAGAS Evaluation Results:")
print(results)


In [ ]:
from ragas import evaluate
from ragas.metrics._answer_relevance import AnswerRelevancyMetric
from langchain_ollama import ChatOllama

#local_model = LocalLlama()
local_model = ChatOllama(model="llama3.2:latest")
test_case = {
   "input": "What is the capital of France?",
   "actual_output": "Nice is the capital of France.",
   "expected_output": "Paris is the capital of France."
}
metric = AnswerRelevancyMetric(model=local_model)
evaluate(test_cases=[test_case], metrics=[metric])

In [18]:

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama, OllamaEmbeddings

# -------------------------------------------------
# 1. Configure local Ollama models
# -------------------------------------------------

ollama_llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0,
)

ollama_embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
)

# Wrap LangChain models for RAGAS
evaluator_llm = LangchainLLMWrapper(ollama_llm)

evaluator_embeddings = LangchainEmbeddingsWrapper(
    ollama_embeddings
)

# -------------------------------------------------
# 2. Prepare RAG evaluation dataset
# -------------------------------------------------
# Each record must contain:
# question, answer, contexts, ground_truth

data = {
    "question": [
        "What is RAG in generative AI?",
        "What is the purpose of an embedding model?",
    ],
    "answer": [
        "RAG stands for Retrieval-Augmented Generation. "
        "It retrieves relevant information and provides it "
        "to an LLM to generate a grounded answer.",
        "An embedding model converts text into numerical "
        "vectors that represent semantic meaning.",
    ],
    "contexts": [
        [
            "Retrieval-Augmented Generation (RAG) retrieves "
            "relevant documents from an external knowledge "
            "source and supplies them to a language model "
            "to help generate an answer."
        ],
        [
            "Embedding models transform text into vectors "
            "that represent semantic information. These "
            "vectors are used for semantic search and "
            "similarity comparisons."
        ],
    ],
    "ground_truth": [
        "RAG retrieves relevant external information and "
        "uses it to help an LLM generate a grounded answer.",
        "An embedding model transforms text into numerical "
        "vectors representing semantic meaning.",
    ],
}

dataset = Dataset.from_dict(data)

# -------------------------------------------------
# 3. Select evaluation metrics
# -------------------------------------------------

metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
]

# -------------------------------------------------
# 4. Evaluate using ONLY local Ollama models
# -------------------------------------------------

results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

# -------------------------------------------------
# 5. Display and save results
# -------------------------------------------------

results_df = results.to_pandas()

print("\nRAGAS Evaluation Results:")
print(results_df.to_string(index=False))

print("\nAverage scores:")
print(results_df[
    [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
    ]
].mean())

results_df.to_csv("ragas_results.csv", index=False)
print("\nResults saved to ragas_results.csv")

C:\Users\ak60492\AppData\Local\Temp\ipykernel_21212\920421562.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\ak60492\AppData\Local\Temp\ipykernel_21212\920421562.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\ak60492\AppData\Local\Temp\ipykernel_21212\920421562.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\ak60492\AppData\Local\Temp\ipykern


RAGAS Evaluation Results:
                                user_input                                                                                                                                                      retrieved_contexts                                                                                                                                  response                                                                                          reference  faithfulness  answer_relevancy  context_precision  context_recall
             What is RAG in generative AI? [Retrieval-Augmented Generation (RAG) retrieves relevant documents from an external knowledge source and supplies them to a language model to help generate an answer.] RAG stands for Retrieval-Augmented Generation. It retrieves relevant information and provides it to an LLM to generate a grounded answer. RAG retrieves relevant external information and uses it to help an LLM generate a grounded answer.           